# Aula 15 — Viagem no tempo: reproduzir e bifurcar

**Objetivo:** usar os checkpoints como máquina do tempo: navegar o histórico,
reproduzir uma execução passada e criar ramos alternativos (forks) sem apagar
o original.

**Pré-requisitos:** Aula 06 (checkpoints por passo) e Aula 13 (`update_state`).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph
print('dependências prontas.')

## 1. Analogia: fita de vídeo com marcadores

Cada execução grava uma **fita**: um marcador por passo. Com ela você:

- **Rebobina e revê** (`get_state_history`): o que aconteceu em cada passo.
- **Repete a cena** (replay): roda de novo a partir de um marcador.
- **Grava final alternativo** (fork): volta ao marcador, troca uma fala e
  segue — a fita original continua intacta na prateleira.

| Fita | No LangGraph | O que é de verdade |
|---|---|---|
| lista de marcadores | **`get_state_history(cfg)`** | do mais novo ao mais antigo |
| marcador | **`checkpoint_id`** em `estado.config` | endereço de um instante |
| repetir a cena | **`stream(None, config_antiga)`** | reexecuta a partir dali |
| final alternativo | **`update_state(config_antiga, {...})`** | cria ramo novo (fork) |
| fita atual | **`get_state(cfg)`** | sempre o ramo mais recente (== histórico[0]) |

In [ ]:
# CÉLULA 1 — executa uma vez e lista a fita (rode e conte)
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel

def multiply(a: int, b: int) -> int:
    """Multiplica a por b."""
    return a * b

# Fake fresco: 1 pedido + 1 frase final (regra das Aulas 06/11/12).
LLM = FakeMessagesListChatModel(responses=[
    AIMessage(content='', id='pedido-1', tool_calls=[
        {'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'c1', 'type': 'tool_call'}]),
    AIMessage(content='6', id='resp-1', tool_calls=[]),
])

def assistant(state: MessagesState) -> dict:
    return {'messages': [LLM.invoke(state['messages'])]}

b = StateGraph(MessagesState)
b.add_node('assistant', assistant)
b.add_node('tools', ToolNode([multiply]))
b.add_edge(START, 'assistant')
b.add_conditional_edges('assistant', tools_condition)
b.add_edge('tools', 'assistant')
viajante = b.compile(checkpointer=MemorySaver())
cfg = {'configurable': {'thread_id': 'fita-1'}}
list(viajante.stream({'messages': [HumanMessage(content='Multiplique 2 e 3', id='hum-1')]}, cfg))

fita = list(viajante.get_state_history(cfg))
print(f'{len(fita)} checkpoints (mais novo primeiro):')
for h in fita:
    print('  next=', h.next, '| msgs=', len(h.values['messages']))
assert len(fita) == 5
assert fita[0].next == () and fita[-1].next == ('__start__',)
print('OK — 5 instantes: entrada + 3 passos + final.')

## 2. Conceito: 3 operações sobre 1 fita

- **Navegar**: `get_state_history(cfg)` devolve do **mais novo** (`[0]` ==
  `get_state(cfg)`) ao **mais antigo** (`__start__`, estado vazio). Cada item
  tem `.values` (estado), `.next` (para onde iria) e `.config` (endereço com
  `checkpoint_id` + `thread_id`).
- **Reproduzir**: `stream(None, config_antiga)` reexecuta **a partir** daquele
  instante. O passado não muda: os checkpoints antigos continuam lá.
- **Bifurcar**: `update_state(config_antiga, {...})` escreve um **novo**
  checkpoint pendurado no antigo (fork). Com `messages`, vale a regra da
  Aula 13: mesmo `id` substitui, id novo anexa. O `stream(None, fork_config)`
  executa o ramo novo — e `get_state(cfg)` passa a mostrar ele.

In [ ]:
# CÉLULA 2 — replay: repete a cena do checkpoint da pergunta
antigo = fita[-2]   # o instante que recebeu a pergunta humana
print('reproduzindo a partir de next =', antigo.next)
for chunk in viajante.stream(None, antigo.config, stream_mode='updates'):
    print(list(chunk.keys()))
print()
print('Leitura: mesmos nós (assistant, tools, assistant), passado intacto.')

## 3. Dissecando o endereço do instante

- `estado.config == {'configurable': {'thread_id': ..., 'checkpoint_id': ...}}`.
  É esse dict que você passa de volta para `stream`/`update_state`.
- `get_state(cfg)` sem `checkpoint_id` = ramo mais recente (atalho para o `[0]`).
- Replay **reexecuta** os nós a partir do instante (não é 'ver gravação'):
  com LLM real, a 2ª passagem pode responder diferente da 1ª — os checkpoints
  guardam estados, não proibem nova computação.

In [ ]:
# CÉLULA 3 — fork: troca a pergunta no passado, segue o ramo novo
fork_cfg = viajante.update_state(antigo.config, {'messages': [
    HumanMessage(content='Multiplique 5 e 3', id='hum-1')]})   # MESMO id: substitui
print('fork criado. Executando o ramo novo:')
for chunk in viajante.stream(None, fork_cfg, stream_mode='updates'):
    print(list(chunk.keys()))
atual = viajante.get_state(cfg).values['messages']
print('estado atual:', [(type(m).__name__, m.content) for m in atual])
assert atual[0].content == 'Multiplique 5 e 3' and atual[0].id == 'hum-1'
print()
print('OK — pergunta trocada no ramo novo (o passado original segue na fita).')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Fork com `id` novo ANEXA em vez de substituir

A regra da Aula 13 vale no passado também: sem o `id` original, a 'correção'
vira mensagem extra — e o ramo novo carrega pergunta duplicada.

In [ ]:
# CÉLULA 4 — fork com id novo: anexa (rode e conte)
cfg4 = {'configurable': {'thread_id': 'fita-2'}}
list(viajante.stream({'messages': [HumanMessage(content='Multiplique 2 e 3', id='hum-1')]}, cfg4))
fita4 = list(viajante.get_state_history(cfg4))
viajante.update_state(fita4[-2].config, {'messages': [
    HumanMessage(content='Multiplique 5 e 3', id='outro-id')]})   # id NOVO
atual4 = viajante.get_state(cfg4).values['messages']
print([(type(m).__name__, m.content) for m in atual4])
assert len(atual4) == 2 and atual4[0].content == 'Multiplique 2 e 3'
print('OK — id novo anexou: agora há DUAS perguntas no ramo.')

### 4b. Replay com LLM real pode divergir do original

Replay **reexecuta** (não mostra gravação). Com Fake determinístico, o traço
repete igual; com LLM real (`temperature>0` ou raciocínio), a 2ª passagem pode
pedir outra tool. Para 'ver o passado sem recomputar', leia `fita[i].values`
direto — sem `stream`.

In [ ]:
# CÉLULA 5 — ler o passado sem executar nada (100% offline)
passado = fita[-2]
print('estado guardado:', [(type(m).__name__, m.content) for m in passado.values['messages']])
print('iria para:', passado.next)
print('endereço:', {k: (v[:8] if k == 'checkpoint_id' else v) for k, v in passado.config['configurable'].items()})
print('OK — leitura pura: nenhum nó rodou nesta célula.')

## 5. Erros clássicos

1. **Achar que replay é 'ver gravação'.** Replay reexecuta os nós (com LLM
   real, pode divergir). Leitura pura é `fita[i].values`, sem `stream`.
2. **Fork sem o `id` original.** Vira anexo duplicado em vez de substituição
   (CÉLULA 4). Para trocar o passado, copie o `id`.
3. **Confundir `get_state` com `history[0]`.** São o mesmo ramo atual — mas
   `history` mostra TODOS os ramos/instantes; `get_state` só o mais recente.
4. **Ordem da fita.** `history[0]` é o MAIS NOVO (não o primeiro). `[-2]` aqui
   foi 'o checkpoint da pergunta' porque a fita tem 5 itens — conte sempre.
5. **Fork sem `stream` depois.** `update_state` só cria o checkpoint; quem
   executa o ramo novo é o `stream(None, fork_config)` seguinte.
6. **Checkpoints em RAM.** `MemorySaver` perde a fita ao reiniciar (Aula 06).
   Viagem no tempo em produção pede checkpointer persistente.

## 6. Exercício guiado — replay auditável da comanda

Situação: a mesa 12 reclama da conta. Você precisa **reproduzir** exatamente
o que aconteceu a partir da pergunta original — sem alterar nada — e listar
cada passo com seu `next`.

In [ ]:
# CÉLULA 6 — exercício guiado (reproduz e audita)
cfg_ex = {'configurable': {'thread_id': 'mesa-12'}}
list(viajante.stream({'messages': [HumanMessage(content='Quanto é 2+2?', id='ped-12')]}, cfg_ex))
fita_ex = list(viajante.get_state_history(cfg_ex))
print('passos na fita:', len(fita_ex))
for h in fita_ex:
    print('  next =', h.next, '| msgs =', len(h.values['messages']))
base = fita_ex[-2]
trac = [list(c.keys()) for c in viajante.stream(None, base.config, stream_mode='updates')]
print('replay:', trac)
assert trac[0] == ['assistant'] and 'tools' in [k for c in trac for k in c]
print('OK — auditoria reproduzida sem tocar no original.')

## 7. Desafio — dois futuros alternativos

A partir da MESMA pergunta (`fita_ex[-2]` do exercício), crie **2 forks**:

1. Ramo A: troca por 'Quanto é 10+10?' (mesmo `id`: substitui).
2. Ramo B: anexa 'com urgência' (id novo: soma).
3. Execute os dois (`stream(None, fork)` em cada) e compare os estados finais.
4. Liste `get_state_history` no fim e identifique os 3 ramos (original + A + B).

- **Pegadinha:** os dois forks pendem do MESMO checkpoint — são irmãos, não
  pai e filho. O `get_state(cfg)` mostra sempre o ÚLTIMO executado.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que `get_state_history` ordena — e o que `[0]` e `[-1]` significam?
- [ ] Replay reexecuta ou mostra gravação? E com LLM real, o que pode mudar?
- [ ] Fork com mesmo `id` vs id novo: o que cada um faz com o passado?
- [ ] Após `update_state` no passado, o que executa o ramo novo?
- [ ] Como ler um checkpoint sem executar nada?
- [ ] Na sua lanchonete, quando você reproduziria vs bifurcaria uma comanda?

**Próxima aula (16 — Paralelização):** fan-out/fan-in de verdade, reducer de
ordenação e RAG paralelo — velocidade com correção.